In [45]:
using Pkg
Pkg.activate("/home/jeremy/flame-solver")
using SparseArrays
using LinearAlgebra
using BenchmarkTools
using LIKWID

using Revise
using Arrhenius

  Activating project at `~/flame-solver`


In [114]:
abstract type Reactions end

struct ForwardReactions{Ty<:AbstractArray,sTy<:AbstractArray}<:Reactions
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    vk::sTy
end

struct ReversibleReactions{Ty<:AbstractArray,sTy<:AbstractArray,vTy<:AbstractVector}<:Reactions
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    product_stoich_coeffs::sTy
    reactant_stoich_coeffs::sTy
    vk::sTy
    vk_sum::vTy
end

struct ThreeBodyReactions{Ty<:AbstractArray,sTy<:AbstractArray,vTy<:AbstractVector}<:Reactions
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    product_stoich_coeffs::sTy
    reactant_stoich_coeffs::sTy
    vk::sTy
    vk_sum::vTy
    efficiency_coeffs::sTy
end

In [3]:
gas = CreateSolution("../mechanism/gri30.yaml",sparseCoeffs=true);
kf = zeros(gas.n_reactions);
kr = zeros(gas.n_reactions);
C = rand(gas.n_species)
wdot = zeros(gas.n_species)
T =900

900

In [119]:
forwardReactions = ForwardReactions(gas.kinetics.Arrhenius_coeffs,
                    gas.kinetics.reactant_orders,
                    sparse(transpose(gas.kinetics.vk)),
                    )

reversibleReactions = ReversibleReactions(
    gas.kinetics.Arrhenius_coeffs,
    gas.kinetics.reactant_orders,
    gas.kinetics.product_stoich_coeffs,
    gas.kinetics.reactant_stoich_coeffs,
    sparse(transpose(gas.kinetics.vk)),
    gas.kinetics.vk_sum
    );

i_tB = gas.kinetics.index_three_body
threeBodyReactions = ThreeBodyReactions(
    gas.kinetics.Arrhenius_coeffs[i_tB,:],
    gas.kinetics.reactant_orders[:,i_tB],
    gas.kinetics.product_stoich_coeffs[:,i_tB],
    gas.kinetics.reactant_stoich_coeffs[:,i_tB],
    sparse(transpose(gas.kinetics.vk[:,i_tB])),
    gas.kinetics.vk_sum[i_tB],
    gas.kinetics.efficiencies_coeffs[:,i_tB]
    );

In [120]:
gas.kinetics.vk_sum[i_tB]

12-element Vector{Float64}:
 -1.0
 -1.0
 -1.0
 -1.0
 -1.0
  1.0
 -1.0
  1.0
 -1.0
  1.0
  1.0
  1.0

In [112]:



function forwardRate!(kf::AbstractVector, arr_coeffs::AbstractArray,T::Real)
    logT = log(T)
    Eexp = (4184.0 / Arrhenius.R / T)
    for i in eachindex(kf) 
        @inbounds  kf[i] = arr_coeffs[i, 1]  * exp(arr_coeffs[i, 2] * logT - arr_coeffs[i, 3] * Eexp)
    end
    #@. kf = @view(arr_coeffs[:, 1])  * exp(@view(arr_coeffs[:, 2]) * logT - @view(arr_coeffs[:, 3]) * Eexp)
end
function kf_Pi_c_rectantOrder!(kf,C,reactant_orders)
    for i in eachindex(kf)
        for j in reactant_orders.colptr[i]:reactant_orders.colptr[i+1]-1
            @inbounds kf[i] *= C[reactant_orders.rowval[j]]^reactant_orders.nzval[j]
        end
    end
end
function vk_kf!(wdot,kf,vk) # TODO use likwid to compare the two methods
    # transposed vk - workflow 
    for isp in eachindex(wdot) 
        for j in vk.colptr[isp]:vk.colptr[isp+1]-1
            wdot[isp] += vk.nzval[j] * kf[vk.rowval[j]]
        end
    end 
    # for ir in eachindex(kf)
    #     for j in vk.colptr[ir]:vk.colptr[ir+1]-1
    #         wdot[vk.rowval[j]] += vk.nzval[j] * kf[ir]
    #     end
    # end
    #mul!(wdot,vk',kf)
    return nothing
end
function wdot_func!(wdot::AbstractVector,kf::AbstractVector,C::AbstractVector,forwardReactions::ForwardReactions ,T::Real)
    wdot .= 0.0
    forwardRate!(kf, forwardReactions.arrhenius_coeffs,T)
    kf_Pi_c_rectantOrder!(kf,C,forwardReactions.reactant_orders)
    vk_kf!(wdot,kf,forwardReactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$kf,$C,$forwardReactions,$T) evals = 200 seconds = 20

BenchmarkTools.Trial: 10000 samples with 200 evaluations.
 Range (min … max):  7.456 μs …  15.087 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     7.615 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   7.973 μs ± 960.102 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▆█▆▆▄▂▂▂▁▁▁▁                            ▂▂                  ▂
  █████████████████████▇██████▇██▇█▇▇██▇▇████▇▆▆▆▇▆▆▅▆▄▃▄▁▄▅▅ █
  7.46 μs      Histogram: log(frequency) by time      12.1 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

# Comparison to old wdot

In [77]:
function wdot_func(reaction, T, C)

    @inbounds _kf = @. @view(reaction.Arrhenius_coeffs[:, 1]) * exp(
        @view(reaction.Arrhenius_coeffs[:, 2]) * log(T) -
        @view(reaction.Arrhenius_coeffs[:, 3]) * (4184.0 / R / T),
    )
    for i = 1:reaction.n_reactions
        @inbounds for j in reaction.i_reactant[i]
            @inbounds _kf[i] *= C[j]^reaction.reactant_orders[j, i]
        end
    end
    

    #if get_qdot
    #    return _kf - _kr
    #end
    return reaction.vk *  (_kf ) #omega_dot_k
end

@benchmark wdot_func($(gas.kinetics),$T,$C)

BenchmarkTools.Trial: 10000 samples with 1 evaluation.
 Range (min … max):  13.336 μs … 264.141 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     13.756 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   14.141 μs ±   2.651 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▁▇█▆▂                                                       
  ▂▄█████▇▆▇▇▆▅▃▂▂▂▁▁▁▁▁▁▂▂▃▃▃▂▂▂▃▃▂▃▄▅▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  13.3 μs         Histogram: frequency by time         16.4 μs <

 Memory estimate: 3.17 KiB, allocs estimate: 2.

In [89]:
wdot_func!(wdot,kf,C,forwardReactions,T)
wdot-wdot_func((gas.kinetics),T,C)

53-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

# reversibleReaction

In [35]:
h_RT = rand(gas.n_species)
s0_R = rand(gas.n_species)
g_RT = h_RT - s0_R
kr = rand(gas.n_reactions);

In [113]:
function kr!(kr, kf,g_RT,vk,vk_sum,T)
    mul!(kr,vk,g_RT) #kr proxies as Δg_RT as intermediate storage'
    logP = log(one_atm / R / T)
    @. kr = kf / exp(-kr + logP * vk_sum)
    return nothing
end



function wdot_func!(wdot::AbstractVector,kf::AbstractVector,kr::AbstractVector,C::AbstractVector,reversibleReactions::ReversibleReactions,g_RT::AbstractVector ,T::Real)
    wdot .= 0.0
    forwardRate!(kf, reversibleReactions.arrhenius_coeffs,T)
    kf_Pi_c_rectantOrder!(kf,C,reversibleReactions.reactant_orders)
    kr!(kr,kf,g_RT,reversibleReactions.vk,reversibleReactions.vk_sum,T)
    vk_kf!(wdot,kf,reversibleReactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$kf,$kr,$C,$reversibleReactions,$g_RT,$T) #evals = 200 seconds = 20

BenchmarkTools.Trial: 10000 samples with 1 evaluation.
 Range (min … max):  10.119 μs … 42.040 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     10.380 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   12.823 μs ±  3.459 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▆                 ▃▃▁▁▁▂▂▂▃▃▃▃▃▃▃▃▃▂▂▁                     ▂
  ██▅▁▄▁▃▁▄▄▃▄▅▄▃▃▃▃▃██████████████████████▇▇▆▅▇█▇▆▆▇██▆▇▆▆▆▆ █
  10.1 μs      Histogram: log(frequency) by time      22.2 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [64]:
transvk = reversibleReactions.vk' |>sparse
@benchmark mul!($kr,$(reversibleReactions.vk'),$g_RT)


BenchmarkTools.Trial: 10000 samples with 168 evaluations.
 Range (min … max):  631.077 ns …   2.673 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     648.196 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   728.290 ns ± 249.842 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▄▂        ▁ ▁    ▁▂▁ ▁▁▁                                  ▁  ▁
  █████▇▇▇▆▅▆███▅▆▆▇███████▇▅▆▄▁▃▃▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁█▇ █
  631 ns        Histogram: log(frequency) by time       2.14 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [65]:
@benchmark mul!($kr,$(transvk),$g_RT)

BenchmarkTools.Trial: 10000 samples with 196 evaluations.
 Range (min … max):  472.684 ns …  2.206 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     490.827 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   499.294 ns ± 53.620 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▁▄▆▇█▇▆▄▁         ▁▂▃▂▂                                    ▂
  ▄▇███████████▇████████████▇▆▆▆▆▆▆▆▅▅▅▅▆▆▄▅▅▆▅▄▃▄▄▄▄▄▄▅▆▅▄▅▄▅ █
  473 ns        Histogram: log(frequency) by time       631 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [15]:
function wdot_func(reaction, T, C,h_RT,s0_R)

    @inbounds _kf = @. @view(reaction.Arrhenius_coeffs[:, 1]) * exp(
        @view(reaction.Arrhenius_coeffs[:, 2]) * log(T) -
        @view(reaction.Arrhenius_coeffs[:, 3]) * (4184.0 / R / T),
    )

    @inbounds ΔS_R = reaction.vk' * s0_R 
    @inbounds ΔH_RT = reaction.vk' * h_RT
    @inbounds Keq =
        @. exp(ΔS_R - ΔH_RT + log(one_atm / R / T) * reaction.vk_sum)
    @inbounds _kr = @. _kf / Keq * reaction.is_reversible

    for i = 1:reaction.n_reactions
        @inbounds for j in reaction.i_reactant[i]
            @inbounds _kf[i] *= C[j]^reaction.reactant_orders[j, i]
        end
        @inbounds for j in reaction.i_product[i]
            @inbounds _kr[i] *= C[j]^reaction.product_stoich_coeffs[j, i]
        end
    end
    #if get_qdot
    #    return _kf - _kr
    #end
    return reaction.vk *  (_kf -_kr ) #omega_dot_k
end
@benchmark wdot_func($gas.kinetics,$T,$C,$h_RT,$s0_R)

BenchmarkTools.Trial: 10000 samples with 1 evaluation.
 Range (min … max):  28.263 μs …  3.064 ms  ┊ GC (min … max): 0.00% … 96.06%
 Time  (median):     29.486 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   31.019 μs ± 50.078 μs  ┊ GC (mean ± σ):  2.74% ±  1.68%

      ▃▇██▇▄▁                                                  
  ▁▁▃▆███████▆▄▃▃▄▄▄▃▂▂▂▁▁▁▁▁▁▁▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▁▁▁ ▂
  28.3 μs         Histogram: frequency by time        36.8 μs <

 Memory estimate: 16.62 KiB, allocs estimate: 8.

# Three body reaction 

In [ ]:
function efficency!( kf,effCoeff,C)
    mul!(kr,vk,g_RT) #kr proxies as Δg_RT as intermediate storage'
    logP = log(one_atm / R / T)
    @. kr = kf / exp(-kr + logP * vk_sum)
    return nothing
end



function wdot_func!(wdot::AbstractVector,kf::AbstractVector,kr::AbstractVector,C::AbstractVector,reversibleReactions::ReversibleReactions,g_RT::AbstractVector ,T::Real)
    wdot .= 0.0
    forwardRate!(kf, reversibleReactions.arrhenius_coeffs,T)
    kf_Pi_c_rectantOrder!(kf,C,reversibleReactions.reactant_orders)
    kr!(kr,kf,g_RT,reversibleReactions.vk,reversibleReactions.vk_sum,T)
    vk_kf!(wdot,kf,reversibleReactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$kf,$kr,$C,$reversibleReactions,$g_RT,$T) #evals = 200 seconds = 20